In [ ]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

class AlternativeDataPipeline:
    """End-to-End Feature Engineering Pipeline for Alternative Credit Data."""

    def __init__(self, current_bs_year=2081):
        self.current_bs_year = current_bs_year

    def _safe_numeric(self, series):
        """Safely cleans commas, 'Rs.', and forces numeric conversion."""
        if series.dtype == "object":
            series = series.astype(str).str.replace(",", "").str.replace("Rs.", "").str.strip()
        return pd.to_numeric(series, errors="coerce").fillna(0).abs()

    def process_profiles_and_loans(self, profiles_df, loans_df):
        """Merges and cleans the base applicant and loan tables."""
        print("⚙️ Processing Profiles & Loans...")

        # Merge base tables on applicant_id
        df = profiles_df.merge(loans_df, on="applicant_id", how="inner", suffixes=("", "_loan"))

        # Defensive column extraction
        base_features = pd.DataFrame({"applicant_id": df["applicant_id"]})

        # Structural categoricals
        if "occupation_en" in df.columns:
            base_features["occupation_en"] = df["occupation_en"].fillna("Unknown")
        if "rural_urban" in df.columns:
            base_features["rural_urban"] = df["rural_urban"].fillna("Unknown")

        # Assets
        base_features["land_area_ropani"] = pd.to_numeric(df.get("land_area_ropani", 0), errors="coerce").fillna(0)

        # Cross-table checks
        base_features["claims_coop_member"] = df.get("cooperative_member", False)

        # Extract Targets if they exist (for training)
        if "income_agent_monthly_est" in df.columns:
            base_features["derived_income_est"] = pd.to_numeric(df["income_agent_monthly_est"], errors="coerce")
        if "income_confidence" in df.columns:
            base_features["income_confidence"] = pd.to_numeric(df["income_confidence"], errors="coerce")

        return base_features

    def process_mobile_money(self, tx_df):
        """Extracts wallet velocity and net cash flows."""
        print("⚙️ Processing Mobile Money Transactions...")
        if tx_df is None or tx_df.empty or "applicant_id" not in tx_df.columns:
            return pd.DataFrame({"applicant_id": []})

        df = tx_df.copy()
        df["amount_clean"] = self._safe_numeric(df.get("amount_nrs", pd.Series([0]*len(df))))

        # Prevent double counting: exclude remittance_receipt from wallet income
        if "transaction_type" in df.columns:
            df = df[df["transaction_type"] != "remittance_receipt"]

        if "direction" in df.columns:
            df["is_credit"] = (df["direction"] == "credit").astype(int)
            df["credit_amt"] = df["amount_clean"] * df["is_credit"]
            df["debit_amt"] = df["amount_clean"] * (1 - df["is_credit"])
        else:
            df["credit_amt"] = df["amount_clean"]
            df["debit_amt"] = 0

        # Aggregations per applicant
        agg_df = df.groupby("applicant_id").agg(
            esewa_tx_count_6months=("amount_clean", "count"),
            wallet_total_credit=("credit_amt", "sum"),
            wallet_total_debit=("debit_amt", "sum")
        ).reset_index()

        # Monthly averages
        agg_df["esewa_net_monthly"] = (agg_df["wallet_total_credit"] - agg_df["wallet_total_debit"]) / 6.0
        # Add new feature: pure cash inflow (often more predictive than net)
        agg_df["wallet_monthly_inflow"] = agg_df["wallet_total_credit"] / 6.0

        return agg_df

    def process_remittances(self, remit_df):
        """Cleans 10x exchange rate noise and calculates discounted remittance."""
        print("⚙️ Processing Remittance Records...")
        if remit_df is None or remit_df.empty or "applicant_id" not in remit_df.columns:
            return pd.DataFrame({"applicant_id": []})

        df = remit_df.copy()

        # Fix exchange rate anomalies (10x noise)
        if "exchange_rate" in df.columns and "foreign_currency_code" in df.columns:
            median_rates = df.groupby("foreign_currency_code")["exchange_rate"].transform("median")
            df["rate_clean"] = np.where(
                df["exchange_rate"] > (median_rates * 3),
                df["exchange_rate"] / 10.0,
                df["exchange_rate"]
            )
        else:
            df["rate_clean"] = 1.0

        amt_fc = pd.to_numeric(df.get("amount_foreign_currency", 0), errors="coerce").fillna(0)
        df["true_amount_nrs"] = amt_fc * df["rate_clean"]

        # Apply fuzzy match trust discount
        match_score = pd.to_numeric(df.get("name_match_score", 1.0), errors="coerce").fillna(0)
        df["trust_factor"] = np.where(match_score >= 0.90, 1.0, np.where(match_score >= 0.80, 0.85, 0.0))
        df["discounted_nrs"] = df["true_amount_nrs"] * df["trust_factor"]

        agg_df = df.groupby("applicant_id").agg(
            remit_count=("discounted_nrs", "count"),
            remit_total=("discounted_nrs", "sum")
        ).reset_index()

        agg_df["remittance_monthly_avg"] = agg_df["remit_total"] / 6.0
        # Regularity score (e.g., getting it almost every month is max score)
        agg_df["remittance_regularity_score"] = (agg_df["remit_count"] / 6.0).clip(upper=1.0)

        return agg_df

    def process_cooperatives(self, members_df, sales_df):
        """Merges cooperative data and extracts tenure and sales."""
        print("⚙️ Processing Cooperative Profiles & Sales...")
        if members_df is None or members_df.empty or "applicant_id" not in members_df.columns:
            return pd.DataFrame({"applicant_id": []})

        # Calculate Tenure
        mem_df = members_df.copy()
        if "membership_year_bs" in mem_df.columns:
            mem_df["coop_tenure_years"] = self.current_bs_year - pd.to_numeric(mem_df["membership_year_bs"], errors="coerce")
            mem_df["coop_tenure_years"] = mem_df["coop_tenure_years"].clip(lower=0).fillna(0)
        else:
            mem_df["coop_tenure_years"] = 0

        # Handle Sales
        if sales_df is not None and not sales_df.empty and "applicant_id" in sales_df.columns:
            s_df = sales_df.copy()
            s_df["sales_clean"] = self._safe_numeric(s_df.get("total_amount_nrs", pd.Series([0]*len(s_df))))
            sales_agg = s_df.groupby("applicant_id")["sales_clean"].sum().reset_index()
            sales_agg.rename(columns={"sales_clean": "total_annual_sales"}, inplace=True)
            mem_df = mem_df.merge(sales_agg, on="applicant_id", how="left")
        else:
            mem_df["total_annual_sales"] = 0

        mem_df["total_annual_sales"] = mem_df["total_annual_sales"].fillna(0)

        # Dataset logic: divide by 6 for the 6-month orchestration window baseline
        mem_df["cooperative_monthly_sales"] = mem_df["total_annual_sales"] / 6.0

        # Add new feature: outstanding loan distress
        if "outstanding_loan_nrs" in mem_df.columns:
            mem_df["coop_debt_burden"] = self._safe_numeric(mem_df["outstanding_loan_nrs"])
        else:
            mem_df["coop_debt_burden"] = 0

        return mem_df[["applicant_id", "coop_tenure_years", "cooperative_monthly_sales", "coop_debt_burden"]]

    def process_utilities(self, util_df):
        """Extracts behavioral credit markers from bill payments."""
        print("⚙️ Processing Utility Payments...")
        if util_df is None or util_df.empty or "applicant_id" not in util_df.columns:
            return pd.DataFrame({"applicant_id": []})

        df = util_df.copy()
        df["bill_clean"] = self._safe_numeric(df.get("bill_amount_nrs", pd.Series([0]*len(df))))
        df["arrears_clean"] = self._safe_numeric(df.get("outstanding_arrears_nrs", pd.Series([0]*len(df))))
        df["on_time_rate"] = pd.to_numeric(df.get("cumulative_on_time_rate", 1.0), errors="coerce").fillna(1.0)

        # Separate electricity for specific rate tracking
        is_elec = (df.get("utility_type", "") == "electricity")
        df["elec_on_time"] = np.where(is_elec, df["on_time_rate"], np.nan)

        agg_df = df.groupby("applicant_id").agg(
            utility_avg_bill_nrs=("bill_clean", "mean"),
            util_arrears_total_nrs=("arrears_clean", "max"), # Max arrears is a safer risk metric than sum
            overall_on_time_rate=("on_time_rate", "mean"),
            elec_on_time_rate=("elec_on_time", lambda x: x.mean(skipna=True))
        ).reset_index()

        agg_df["elec_on_time_rate"] = agg_df["elec_on_time_rate"].fillna(agg_df["overall_on_time_rate"])

        # New feature: Arrears Flag (binary risk indicator)
        agg_df["has_utility_arrears"] = (agg_df["util_arrears_total_nrs"] > 0).astype(int)

        return agg_df

    def build_master_dataset(self, profiles, loans, transactions, remittances, coop_members, coop_sales, utilities):
        """Orchestrates the pipeline and joins all features safely."""
        print("🚀 Initiating Master Pipeline Build...")

        # 1. Process individual streams
        master_df = self.process_profiles_and_loans(profiles, loans)
        tx_features = self.process_mobile_money(transactions)
        rem_features = self.process_remittances(remittances)
        coop_features = self.process_cooperatives(coop_members, coop_sales)
        util_features = self.process_utilities(utilities)

        # 2. Merge sequentially (Left Join to preserve all applicants)
        dfs_to_merge = [tx_features, rem_features, coop_features, util_features]
        for df in dfs_to_merge:
            if not df.empty:
                master_df = master_df.merge(df, on="applicant_id", how="left")

        # 3. Defensive Fill NAs for newly created columns
        fill_zero_cols = [
            "esewa_tx_count_6months", "esewa_net_monthly", "wallet_monthly_inflow",
            "remittance_monthly_avg", "remittance_regularity_score", "coop_tenure_years",
            "cooperative_monthly_sales", "coop_debt_burden", "utility_avg_bill_nrs",
            "util_arrears_total_nrs", "has_utility_arrears"
        ]
        for col in fill_zero_cols:
            if col in master_df.columns:
                master_df[col] = master_df[col].fillna(0)

        fill_one_cols = ["overall_on_time_rate", "elec_on_time_rate"]
        for col in fill_one_cols:
            if col in master_df.columns:
                master_df[col] = master_df[col].fillna(1.0)

        # 4. Feature Engineering: Cross-Table Logic

        # A) Income Signal Count (How many distinct streams generated money?)
        master_df["income_signal_count"] = (
            (master_df.get("wallet_monthly_inflow", 0) > 0).astype(int) +
            (master_df.get("remittance_monthly_avg", 0) > 0).astype(int) +
            (master_df.get("cooperative_monthly_sales", 0) > 0).astype(int)
        )

        # B) Detect Ghost Cooperative Members
        if "claims_coop_member" in master_df.columns and "coop_tenure_years" in master_df.columns:
            master_df["is_ghost_member"] = (
                (master_df["claims_coop_member"] == True) &
                (master_df["coop_tenure_years"] == 0)
            ).astype(int)
            master_df.drop(columns=["claims_coop_member"], inplace=True)

        print("✅ Master Dataset Built Successfully!")
        return master_df

# =====================================================================
# HOW TO EXECUTE THE PIPELINE IN YOUR NOTEBOOK
# =====================================================================

# 1. Load your raw files (Note: Parquet for massive tables per dataset docs)
profiles_df = pd.read_csv("applicant_profiles.csv")
loans_df = pd.read_csv("loan_applications.csv")
tx_df = pd.read_parquet("mobile_money_transactions.parquet")
remit_df = pd.read_parquet("remittance_records.parquet")
coop_members_df = pd.read_csv("cooperative_members.csv")
coop_sales_df = pd.read_csv("cooperative_sales.csv")
util_df = pd.read_parquet("utility_payments.parquet")

# 2. Initialize and Run the Pipeline
pipeline = AlternativeDataPipeline(current_bs_year=2081)

train_ready_df = pipeline.build_master_dataset(
    profiles=profiles_df,
    loans=loans_df,
    transactions=tx_df,
    remittances=remit_df,
    coop_members=coop_members_df,
    coop_sales=coop_sales_df,
    utilities=util_df
)

# 3. View your robust features
print(train_ready_df.head())
train_ready_df.to_csv("master_training_features.csv", index=False)

🚀 Initiating Master Pipeline Build...
⚙️ Processing Profiles & Loans...
⚙️ Processing Mobile Money Transactions...
⚙️ Processing Remittance Records...
⚙️ Processing Cooperative Profiles & Sales...
⚙️ Processing Utility Payments...
✅ Master Dataset Built Successfully!
  applicant_id      occupation_en rural_urban  land_area_ropani  \
0    AP-000001       Small Trader       rural                 6   
1    AP-000002            Artisan       rural                 7   
2    AP-000003     Business Owner       urban                 2   
3    AP-000004  Daily Wage Worker  semi_urban                17   
4    AP-000005             Farmer  semi_urban                 6   

   derived_income_est  income_confidence  esewa_tx_count_6months  \
0               42667              0.689                    23.0   
1               11316              0.407                    31.0   
2               16498              0.330                    27.0   
3               17864              0.330                 

In [ ]:
train_ready_df.shape

(100000, 25)

# **TRAINING/TESTING ON ABOVE MADE DATASET**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split


def run_rigorous_pipeline_evaluation(engineered_df):
    """Evaluates the model using the custom DataFrame generated from your raw table pipeline."""
    print("🔄 Initializing rigorous split validation on custom engineered dataset...")
    df = engineered_df.copy()

    # 1. Define ALL features (including your newly engineered context markers)
    numeric_features = [
        "remittance_monthly_avg",
        "esewa_net_monthly",
        "cooperative_monthly_sales",
        "utility_avg_bill_nrs",
        "land_area_ropani",
        "income_signal_count",
        "remittance_regularity_score",
        "esewa_tx_count_6months",
        "overall_on_time_rate",
        "util_arrears_total_nrs",
        "coop_tenure_years",
        # --- NEW PIPELINE FEATURES ADDED BELOW ---
        "wallet_monthly_inflow",
        "coop_debt_burden",
        "elec_on_time_rate",
        "has_utility_arrears",
        "is_ghost_member",
    ]
    categorical_features = ["occupation_en", "rural_urban"]

    # Align types and handle anomalies defensively
    for col in numeric_features:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
        else:
            df[col] = 0

    for col in categorical_features:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown").astype("category")
        else:
            df[col] = "Unknown"
            df[col] = df[col].astype("category")

    all_features = numeric_features + categorical_features

    # 2. Handle Partitioning Logic
    if "data_split" in df.columns:
        print("💡 Found pre-existing 'data_split' column. Using structural partitions...")
        train_mask = df["data_split"] == "train"
        val_mask = df["data_split"] == "val"
        test_mask = df["data_split"] == "test"

        if not val_mask.any() and not test_mask.any():
            print("⚠️ Data split labels sparse. Generating randomized fallbacks...")
            train_df, test_df = train_test_split(
                df, test_size=0.30, random_state=42, stratify=df["occupation_en"]
            )
            val_df, test_df = train_test_split(
                test_df,
                test_size=0.50,
                random_state=42,
                stratify=test_df["occupation_en"],
            )
        else:
            train_df = df[train_mask]
            val_df = df[val_mask]
            test_df = df[test_mask]
    else:
        print(
            "🎲 No pre-defined split column found in raw tables. Generating a"
            " stratified 70/15/15 Split..."
        )
        train_df, temp_df = train_test_split(
            df, test_size=0.30, random_state=42, stratify=df["occupation_en"]
        )
        val_df, test_df = train_test_split(
            temp_df,
            test_size=0.50,
            random_state=42,
            stratify=temp_df["occupation_en"],
        )

    # Separate X arrays and targets
    X_train, y_train_inc, y_train_conf = (
        train_df[all_features],
        train_df["derived_income_est"],
        train_df["income_confidence"],
    )
    X_val, y_val_inc, y_val_conf = (
        val_df[all_features],
        val_df["derived_income_est"],
        val_df["income_confidence"],
    )
    X_test, y_test_inc, y_test_conf = (
        test_df[all_features],
        test_df["derived_income_est"],
        test_df["income_confidence"],
    )

    print(f"📊 Dataset Sizes -> Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

    # 3. Model Training
    print("\n🚀 Training estimators on the isolated training split...")
    model_income = HistGradientBoostingRegressor(
        loss="squared_error",
        categorical_features="from_dtype",
        max_iter=150,
        random_state=42,
    )
    model_income.fit(X_train, y_train_inc)

    model_confidence = HistGradientBoostingRegressor(
        loss="squared_error",
        categorical_features="from_dtype",
        max_iter=100,
        random_state=42,
    )
    model_confidence.fit(X_train, y_train_conf)

    # 4. Generate & Post-Process Predictions
    preds_train_inc = model_income.predict(X_train).clip(3000, 200000).round(0)
    preds_val_inc = model_income.predict(X_val).clip(3000, 200000).round(0)
    preds_test_inc = model_income.predict(X_test).clip(3000, 200000).round(0)

    preds_train_cf = model_confidence.predict(X_train).clip(0.05, 0.97).round(2)
    preds_val_cf = model_confidence.predict(X_val).clip(0.05, 0.97).round(2)
    preds_test_cf = model_confidence.predict(X_test).clip(0.05, 0.97).round(2)

    # 5. Output Partitioned Metric Matrix
    print("\n🔬 VERIFICATION PERFORMANCE MATRIX:")
    print("=" * 65)

    print(f"🔹 TRAINING SPLIT:")
    print(
        f"  ↳ Income RMSE: NRs"
        f" {root_mean_squared_error(y_train_inc, preds_train_inc):.2f} | MAE:"
        f" NRs {mean_absolute_error(y_train_inc, preds_train_inc):.2f}"
    )
    print(
        f"  ↳ Confidence RMSE:"
        f" {root_mean_squared_error(y_train_conf, preds_train_cf):.4f} | MAE:"
        f" {mean_absolute_error(y_train_conf, preds_train_cf):.4f}"
    )
    print("-" * 65)

    print(f"🔹 VALIDATION SPLIT (UNSEEN DEVELOPMENT):")
    val_rmse = root_mean_squared_error(y_val_inc, preds_val_inc)
    print(f"  ↳ Income RMSE: NRs {val_rmse:.2f} | MAE: NRs {mean_absolute_error(y_val_inc, preds_val_inc):.2f}")
    print(
        f"  ↳ Confidence RMSE:"
        f" {root_mean_squared_error(y_val_conf, preds_val_cf):.4f} | MAE:"
        f" {mean_absolute_error(y_val_conf, preds_val_cf):.4f}"
    )
    print("-" * 65)

    print(f"🔹 HOLDOUT TEST SPLIT (FINAL UNSEEN TARGET):")
    test_rmse = root_mean_squared_error(y_test_inc, preds_test_inc)
    print(
        f"  ↳ Income RMSE: NRs {test_rmse:.2f} | MAE: NRs"
        f" {mean_absolute_error(y_test_inc, preds_test_inc):.2f}"
    )
    print(
        f"  ↳ Confidence RMSE:"
        f" {root_mean_squared_error(y_test_conf, preds_test_cf):.4f} | MAE:"
        f" {mean_absolute_error(y_test_conf, preds_test_cf):.4f}"
    )
    print("=" * 65)

    if test_rmse < 5000:
        print(
            "🏆 SUCCESS: The model generalizes securely! Unseen Test RMSE is"
            " safely below the NRs 5k limit."
        )
    else:
        print(
            "⚠️ WARNING: Generalization error is too high. Regularization parameter adjustments required."
        )

    # 6. Append final orchestration predictions back to the input frame
    df["pred_income_est"] = model_income.predict(df[all_features]).clip(3000, 200000).round(0)
    df["pred_confidence"] = model_confidence.predict(df[all_features]).clip(0.05, 0.97).round(2)

    return df[["applicant_id", "pred_income_est", "pred_confidence"]]


# =====================================================================
# HOW TO RUN IN YOUR NOTEBOOK CELL
# =====================================================================
# Simply pass your 'train_ready_df' object directly from your feature factory:
final_predictions_df = run_rigorous_pipeline_evaluation(train_ready_df)

🔄 Initializing rigorous split validation on custom engineered dataset...
🎲 No pre-defined split column found in raw tables. Generating a stratified 70/15/15 Split...
📊 Dataset Sizes -> Train: 70000 | Val: 15000 | Test: 15000

🚀 Training estimators on the isolated training split...

🔬 VERIFICATION PERFORMANCE MATRIX:
🔹 TRAINING SPLIT:
  ↳ Income RMSE: NRs 3131.39 | MAE: NRs 1105.15
  ↳ Confidence RMSE: 0.0408 | MAE: 0.0334
-----------------------------------------------------------------
🔹 VALIDATION SPLIT (UNSEEN DEVELOPMENT):
  ↳ Income RMSE: NRs 3602.43 | MAE: NRs 1211.68
  ↳ Confidence RMSE: 0.0421 | MAE: 0.0341
-----------------------------------------------------------------
🔹 HOLDOUT TEST SPLIT (FINAL UNSEEN TARGET):
  ↳ Income RMSE: NRs 3697.36 | MAE: NRs 1214.71
  ↳ Confidence RMSE: 0.0412 | MAE: 0.0336
🏆 SUCCESS: The model generalizes securely! Unseen Test RMSE is safely below the NRs 5k limit.


In [ ]:
import numpy as np
import pandas as pd

def build_refined_income_agent(profiles_df, tx_df, remit_df, coop_sales_df, util_df):
    """
    Optimized Deterministic Rule-Based Income & Confidence Engine.
    Reverse-engineers the likely mathematical baseline for the Hackathon.
    """
    print("⚙️ Initializing Mathematical Income Aggregation Pipeline...")

    # 1. Base Matrix
    income_features = pd.DataFrame(index=profiles_df["applicant_id"])

    # -------------------------------------------------------------------------
    # STREAM 1: MOBILE MONEY (Wallet Credits)
    # -------------------------------------------------------------------------
    tx = tx_df.copy()
    # Clean string noise safely
    tx["amount_clean"] = tx["amount_nrs"].astype(str).str.replace(r"[^\d.]", "", regex=True)
    tx["amount_clean"] = pd.to_numeric(tx["amount_clean"], errors="coerce").fillna(0)

    # Filter out remittances (handled separately) and grab credits
    wallet_credits = tx[(tx["direction"] == "credit") & (tx["transaction_type"] != "remittance_receipt")]

    # Calculate Monthly Average & Volatility (Standard Deviation)
    wallet_grp = wallet_credits.groupby("applicant_id")["amount_clean"]
    income_features["i_wallet_avg"] = (wallet_grp.sum() / 6.0).reindex(income_features.index, fill_value=0)
    income_features["i_wallet_std"] = wallet_grp.std().reindex(income_features.index, fill_value=0)

    # -------------------------------------------------------------------------
    # STREAM 2: REMITTANCES
    # -------------------------------------------------------------------------
    rem = remit_df.copy()

    # Fix impossible exchange rates (Dataset Noise)
    median_rates = rem.groupby("foreign_currency_code")["exchange_rate"].transform("median")
    rem["exchange_rate_fixed"] = np.where(
        rem["exchange_rate"] > (median_rates * 3),
        rem["exchange_rate"] / 10.0,
        rem["exchange_rate"]
    )

    # Calculate base received amount
    rem["amount_nrs_clean"] = rem["amount_foreign_currency"] * rem["exchange_rate_fixed"]

    # Fuzzy match discount: If name match is low, the transfer is risky to count as personal income
    rem["discount_factor"] = np.where(rem["name_match_score"] >= 0.90, 1.0,
                             np.where(rem["name_match_score"] >= 0.80, 0.85, 0.50))
    rem["discounted_amount"] = rem["amount_nrs_clean"] * rem["discount_factor"]

    rem_grp = rem.groupby("applicant_id")["discounted_amount"]
    income_features["i_remit_avg"] = (rem_grp.sum() / 6.0).reindex(income_features.index, fill_value=0)
    income_features["i_remit_std"] = rem_grp.std().reindex(income_features.index, fill_value=0)

    # -------------------------------------------------------------------------
    # STREAM 3: COOPERATIVE SALES
    # -------------------------------------------------------------------------
    coop = coop_sales_df.copy()
    coop["total_amount_clean"] = pd.to_numeric(coop["total_amount_nrs"], errors="coerce").fillna(0)

    # Ghost Membership Check: Drop sales if profile says they aren't a member
    valid_members = profiles_df[profiles_df["cooperative_member"] == True]["applicant_id"]
    coop = coop[coop["applicant_id"].isin(valid_members)]

    coop_grp = coop.groupby("applicant_id")["total_amount_clean"]
    income_features["i_coop_avg"] = (coop_grp.sum() / 6.0).reindex(income_features.index, fill_value=0)
    income_features["i_coop_std"] = coop_grp.std().reindex(income_features.index, fill_value=0)

    # -------------------------------------------------------------------------
    # FINAL ESTIMATE CALCULATION
    # -------------------------------------------------------------------------
    income_features["income_agent_monthly_est"] = (
        income_features["i_wallet_avg"] +
        income_features["i_remit_avg"] +
        income_features["i_coop_avg"]
    )

    # Enforce operational bounds (NRB/Dataset rules)
    income_features["income_agent_monthly_est"] = (
        income_features["income_agent_monthly_est"]
        .clip(lower=3000, upper=200000)
        .round(0)
    )

    # -------------------------------------------------------------------------
    # DYNAMIC CONFIDENCE SCORING
    # -------------------------------------------------------------------------
    # 1. Sparsity (How many distinct income streams does the applicant have?)
    streams = ["i_wallet_avg", "i_remit_avg", "i_coop_avg"]
    n_active_streams = (income_features[streams] > 0).sum(axis=1)
    sparsity_score = n_active_streams / 3.0

    # 2. Consistency (Inverse of coefficient of variation)
    total_std = income_features[["i_wallet_std", "i_remit_std", "i_coop_std"]].fillna(0).mean(axis=1)
    # Avoid division by zero
    cv = np.where(income_features["income_agent_monthly_est"] > 0,
                  total_std / income_features["income_agent_monthly_est"], 1)
    consistency_score = np.clip(1.0 - cv, 0, 1)

    # 3. Penalties
    util_perf = util_df.groupby("applicant_id")["cumulative_on_time_rate"].mean().reindex(income_features.index, fill_value=1.0)
    p_utility = np.where(util_perf < 0.50, 0.15, 0.0)

    has_coop_rows = profiles_df["applicant_id"].isin(coop_sales_df["applicant_id"])
    p_ghost = np.where((profiles_df["cooperative_member"] == True) & (~has_coop_rows), 0.20, 0.0)

    # Mathematical Formula for Confidence
    w1, w2 = 0.5, 0.5
    income_features["income_confidence"] = (w1 * sparsity_score) + (w2 * consistency_score) - p_utility - p_ghost

    # Enforce strict 0.05 to 0.97 boundaries
    income_features["income_confidence"] = income_features["income_confidence"].clip(lower=0.05, upper=0.97).round(2)

    return income_features[["income_agent_monthly_est", "income_confidence"]].reset_index()

In [ ]:
# 1. Load your raw files (Note: Parquet for massive tables per dataset docs)
profiles_df_raw = pd.read_csv("applicant_profiles.csv")
tx_df_raw = pd.read_parquet("mobile_money_transactions.parquet")
remit_df_raw = pd.read_parquet("remittance_records.parquet")
coop_sales_df_raw = pd.read_csv("cooperative_sales.csv")
util_df_raw = pd.read_parquet("utility_payments.parquet")

# 2. Execute the build_refined_income_agent function
refined_agent_output = build_refined_income_agent(
    profiles_df_raw,
    tx_df_raw,
    remit_df_raw,
    coop_sales_df_raw,
    util_df_raw
)

print("\nRefined Agent Output (first 5 rows):")
display(refined_agent_output.head())

⚙️ Initializing Mathematical Income Aggregation Pipeline...

Refined Agent Output (first 5 rows):


,applicant_id,income_agent_monthly_est,income_confidence
0,AP-000001,192956.0,0.97
1,AP-000002,6696.0,0.76
2,AP-000003,3000.0,0.45
3,AP-000004,3000.0,0.50
4,AP-000005,3000.0,0.67


```markdown
Now, let's compare the output of the `build_refined_income_agent` with the actual `derived_income_est` and `income_confidence` from the `train_ready_df` to calculate the error metrics.
```

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Merge the refined agent's predictions with the actual values from train_ready_df
# Ensure 'train_ready_df' has the 'derived_income_est' and 'income_confidence' columns
comparison_df_refined = refined_agent_output.merge(
    train_ready_df[['applicant_id', 'derived_income_est', 'income_confidence']],
    on='applicant_id',
    suffixes=('_predicted', '_actual')
)

# Calculate RMSE and MAE for Income Estimation
rmse_income_refined = root_mean_squared_error(comparison_df_refined['derived_income_est'],
                                              comparison_df_refined['income_agent_monthly_est'])
mae_income_refined = mean_absolute_error(comparison_df_refined['derived_income_est'],
                                             comparison_df_refined['income_agent_monthly_est'])

# Calculate RMSE and MAE for Confidence Score
rmse_confidence_refined = root_mean_squared_error(comparison_df_refined['income_confidence_actual'],
                                                  comparison_df_refined['income_confidence_predicted'])
mae_confidence_refined = mean_absolute_error(comparison_df_refined['income_confidence_actual'],
                                                 comparison_df_refined['income_confidence_predicted'])

print("\n--- Evaluation of Refined Income Agent ---")
print(f"Income Estimate RMSE: NRs {rmse_income_refined:.2f}")
print(f"Income Estimate MAE : NRs {mae_income_refined:.2f}")
print(f"Confidence Score RMSE: {rmse_confidence_refined:.4f}")
print(f"Confidence Score MAE : {mae_confidence_refined:.4f}")

# Optionally, display some rows with the comparison
print("\nSample comparison of actual vs. predicted (first 5 rows):")
display(comparison_df_refined[[
    'applicant_id',
    'derived_income_est', 'income_agent_monthly_est',
    'income_confidence_actual', 'income_confidence_predicted'
]].head())


--- Evaluation of Refined Income Agent ---
Income Estimate RMSE: NRs 22512.70
Income Estimate MAE : NRs 10762.65
Confidence Score RMSE: 0.2100
Confidence Score MAE : 0.1829

Sample comparison of actual vs. predicted (first 5 rows):


,applicant_id,derived_income_est,income_agent_monthly_est,income_confidence_actual,income_confidence_predicted
0,AP-000001,42667,192956.0,0.689,0.97
1,AP-000002,11316,6696.0,0.407,0.76
2,AP-000003,16498,3000.0,0.330,0.45
3,AP-000004,17864,3000.0,0.330,0.50
4,AP-000005,14102,3000.0,0.330,0.67
